# AudioSR T4 / TensorRT — изолированная проверка

Маленький блокнот только для проверки ускорения AudioSR на NVIDIA T4. Он не меняет рабочий Gradio и не вливает ничего в `main`.

**Порядок:** включить T4 → выполнить ячейки сверху вниз → загрузить один аудиофайл → получить сравнение PyTorch/TensorRT, VRAM, SNR и WAV-файлы.


In [ ]:
import platform
import subprocess
import sys

import torch

print('Python:', sys.version.split()[0])
print('Ubuntu:', platform.platform())
print('PyTorch:', torch.__version__)
print('PyTorch CUDA:', torch.version.cuda)
print('CUDA доступна:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'Нужен Colab GPU runtime: Runtime → Change runtime type → T4 GPU.'
    )
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
if 'T4' not in gpu:
    print(
        'Предупреждение: probe рассчитан прежде всего на T4; '
        'сейчас выдана другая GPU.'
    )
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO = Path('/content/audio-restoration-colab')
BRANCH = 'agent/audiosr-t4-tensorrt'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    'https://github.com/egor125552/audio-restoration-colab.git', str(REPO)
], check=True)
print('Код готов:', REPO)


In [ ]:
import subprocess

CACHE = '/content/audio-restoration-models'
subprocess.run([
    'bash', str(REPO / 'scripts/prepare_audiosr_t4.sh'), CACHE
], check=True)
PROBE_PYTHON = str(Path(CACHE) / 'envs/audiosr_trt/bin/python')
print('TensorRT-среда готова:', PROBE_PYTHON)


In [ ]:
from pathlib import Path

from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Файл не выбран.')
INPUT_FILE = Path('/content') / next(iter(uploaded))
print('Вход:', INPUT_FILE)


In [ ]:
import subprocess
from pathlib import Path

RESULTS = Path('/content/audiosr-t4-results')
RESULTS.mkdir(parents=True, exist_ok=True)
command = [
    PROBE_PYTHON,
    str(REPO / 'scripts/probe_audiosr_t4.py'),
    '--input', str(INPUT_FILE),
    '--output-dir', str(RESULTS),
    '--mode', 'basic',
    '--steps', '30',
    '--guidance', '3.5',
    '--seed', '42',
    '--runs', '2',
]
print('Запускаю безопасный probe на первых 5.12 секунды аудио…')
subprocess.run(command, check=True)


In [ ]:
from IPython.display import Audio, display

print('Готово. WAV для сравнения:')
for path in sorted(RESULTS.glob('*.wav')):
    print('-', path.name)
    display(Audio(filename=str(path)))
